In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_extract, col, when, to_date

In [2]:
problem_name = os.path.basename(os.path.abspath("."))

In [3]:
spark = SparkSession.builder.appName(problem_name).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 23:50:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = spark.read\
.option("header", "true")\
.format("csv")\
.load("files/people_data.txt")

In [5]:
df.show()

+-----+---+----------+
| Name| ID|       DOB|
+-----+---+----------+
| John|  1|1990-05-25|
|Alice|  2|15-08-1985|
|  Bob|  3|2001-11-10|
| Mary|  4|12-07-1978|
+-----+---+----------+



In [6]:
result_df = df.withColumn("dbo_yyy", regexp_extract("DOB", "\\d{4}-\\d{2}-\\d{2}", 0))\
.withColumn("dbo_dd", regexp_extract("DOB", "\\d{2}-\\d{2}-\\d{4}", 0))\
.withColumn("Correction_Status", 
           when(col("dbo_yyy").isin("", None), "DOB corrected").otherwise("No change required")
           )\
.withColumn("DOB", 
           when(col("Correction_Status")=="DOB corrected", to_date("DOB", "dd-MM-yyyy"))\
            .otherwise(col("DOB"))
           )\
.select("Name", "ID", "DOB", "Correction_Status")

In [7]:
result_df.show()

+-----+---+----------+------------------+
| Name| ID|       DOB| Correction_Status|
+-----+---+----------+------------------+
| John|  1|1990-05-25|No change required|
|Alice|  2|1985-08-15|     DOB corrected|
|  Bob|  3|2001-11-10|No change required|
| Mary|  4|1978-07-12|     DOB corrected|
+-----+---+----------+------------------+



In [8]:
spark.stop()